# Validação de Pneus — ApexSim (M1)

Este notebook valida as funções puras de `apexsim/physics.py` contra:
1. As curvas reais extraídas pelo Octávio (`curvas_degradacao`) — **AQUI SUBSTITUÍDAS POR UM PLACEHOLDER SINTÉTICO**, já que o dataset do Octávio ainda não foi integrado ao repositório. O arquivo `reference_octavio_placeholder.json` está marcado com `nota: PLACEHOLDER` e deve ser trocado assim que `curvas_degradacao` (sem. 4) estiver disponível.
2. Os critérios de aceite do roadmap (M1):
   - Macio degrada mais rápido que Médio, que degrada mais rápido que Duro.
   - Carro cheio é mais lento que carro vazio, na ordem de ~0,3 s por 10 kg.

Fundamentação matemática completa: `docs/fisica.md`, Seções 1 e 2.

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

from apexsim.physics import tyre_grip, tyre_time_loss, fuel_time_delta, TYRE_TABLE

with open("reference_octavio_placeholder.json") as f:
    ref = json.load(f)

print(ref["nota"])
ages = ref["ages_laps"]
compounds = ["soft", "medium", "hard"]

## 1. Curva de grip por composto: modelo vs. referência (placeholder)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5), sharey=True)
colors = {"soft": "tab:red", "medium": "tab:orange", "hard": "tab:gray"}

for ax, c in zip(axes, compounds):
    model_curve = [tyre_grip(c, a) for a in ages]
    ref_curve = ref["grip"][c]
    ax.plot(ages, model_curve, label="modelo (physics.py)", color=colors[c], lw=2)
    ax.plot(ages, ref_curve, "--", label="referência (placeholder Octávio)", color="black", alpha=0.6)
    ax.set_title(c.capitalize())
    ax.set_xlabel("Idade do pneu (voltas)")
    ax.grid(alpha=0.3)

axes[0].set_ylabel("Grip normalizado")
axes[0].legend(fontsize=8)
plt.suptitle("Grip vs. idade — modelo ApexSim vs. curva de referência (40 voltas)")
plt.tight_layout()
plt.savefig("validacao_pneus_grip.png", dpi=130)
plt.show()

## 2. Ordenação de degradação entre compostos

In [ ]:
age_check = 25
losses = {c: tyre_time_loss(c, age_check) for c in compounds}
print(f"Perda de tempo na volta {age_check} (idade do pneu):")
for c, v in losses.items():
    print(f"  {c:>6}: {v:.3f} s")

assert losses["soft"] > losses["medium"] > losses["hard"], "Ordem de degradação incorreta!"
print("\nOK: Soft > Medium > Hard em perda de tempo (macio degrada mais rápido).")

## 3. Penalidade de combustível (~0,3 s por 10 kg)

In [ ]:
fuel_levels = np.arange(0, 111, 10)
deltas = [fuel_time_delta(m) for m in fuel_levels]

plt.figure(figsize=(6, 4))
plt.plot(fuel_levels, deltas, marker="o")
plt.xlabel("Combustível a bordo (kg)")
plt.ylabel("Delta de tempo de volta (s)")
plt.title("Penalidade de tempo por massa de combustível")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("validacao_combustivel.png", dpi=130)
plt.show()

delta_10kg = fuel_time_delta(10) - fuel_time_delta(0)
print(f"Delta por 10kg: {delta_10kg:.3f} s (esperado ~0.3s, faixa aceita 0.2-0.5s)")
assert 0.2 <= delta_10kg <= 0.5
print("OK: penalidade de combustível na ordem de grandeza esperada.")

## 4. Resumo dos critérios "Pronto quando" (M1)

- [x] Funções puras, sem dependência do motor de eventos (`tyre_grip`, `fuel_time_delta` em `apexsim/physics.py`).
- [x] Testes automatizados em `tests/test_physics.py` (11 casos, todos passando).
- [x] Este notebook plota as curvas geradas vs. as curvas de referência — **pendente**: trocar o placeholder sintético pelas curvas reais do Octávio assim que `curvas_degradacao` estiver disponível (dependência declarada no roadmap, sem. 4).
- [x] Macio degrada mais rápido que Médio, que degrada mais rápido que Duro.
- [x] Carro cheio é mais lento que carro vazio, na ordem de ~0,3 s por 10 kg.

### Próximo passo
Assim que o Octávio publicar `curvas_degradacao`, recarregar este notebook substituindo `reference_octavio_placeholder.json` pelo dataset real e recalibrar `TYRE_TABLE` em `apexsim/physics.py` (Seção 1.3 de `docs/fisica.md`) via regressão sobre os dados reais.